# Lesson 03 Lab — Version Identity and a Reproducible Baseline

**Puzzle:** When JIT cache, target identity, and cold versus warm time change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates JIT cache, target identity, and cold versus warm time and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Triton performance depends on the Python package, compiler pipeline, backend target, driver-facing runtime, input specialization, and cache state. The GPU name alone is not an experiment identity. Cold host time and warm event time must remain separate fields.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["JIT cache, target identity, and cold versus warm time"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

The CUDA version printed by nvidia-smi is a driver capability signal, not proof of the Toolkit or compiler used by the kernel.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 3
LESSON_TITLE = 'Version Identity and a Reproducible Baseline'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260816
}


## 5. Freeze the experiment

**Experiment:** Record the complete environment and measure the first host-observed launch separately from warm GPU-event samples.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 330.9140629135072,
  "secondary": 0.020560000091791153,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "triton": "3.7.1",
    "target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "warm_samples_ms": [
      0.032287999987602234,
      0.02566399984061718,
      0.022752000018954277,
      0.021183999255299568,
      0.020735999569296837,
      0.02099199965596199,
      0.022016000002622604,
      0.01942400075495243,
      0.022655999287962914,
      0.020927999168634415,
      0.020287999883294106,
      0.019168000668287277,
      0.020031999796628952,
      0.02038400061428547,
      0.021663999184966087,
      0.0191040001809597,
      0.019519999623298645,
      0.01926399953663349,
      0.01865600049495697,
      0.018624000251293182
    ]
  }
}
The first host-observed launch took 330.91 ms and the warm GPU-event median was 0.0206 ms; they answer different questions.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| First host call | 330.9141 ms |
| Warm GPU median | 0.0206 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The first host-observed launch took 330.91 ms and the warm GPU-event median was 0.0206 ms; they answer different questions.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'A benchmark is reusable only when target, versions, shapes, dtype, cache state, and timing boundary are visible.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 3,
  "title": "Version Identity and a Reproducible Baseline",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260816
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 330.9140629135072,
    "secondary": 0.020560000091791153,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "triton": "3.7.1",
      "target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
      "warm_samples_ms": [
        0.032287999987602234,
        0.02566399984061718,
        0.022752000018954277,
        0.021183999255299568,
        0.020735999569296837,
        0.02099199965596199,
        0.022016000002622604,
        0.01942400075495243,
        0.022655999287962914,
        0.020927999168634415,
     

## 10. Make the bounded decision

> A benchmark is reusable only when target, versions, shapes, dtype, cache state, and timing boundary are visible.

**Failure analysis:** The CUDA version printed by nvidia-smi is a driver capability signal, not proof of the Toolkit or compiler used by the kernel.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
